In [1]:
import pandas as pd
import plotly.express as px

In [2]:
# 1. Load the data
df = pd.read_csv(r"D:\Saqib\SkillSetExpand\WA_Solar_Grid_Optimizer\data\raw/perth_solar_raw.csv")

In [3]:
# 2. Calculate Correlation Matrix
# We want to see how 'cloud_cover' and 'temperature' relate to 'direct_normal_irradiance'
corr_matrix = df.drop(columns=['time']).corr()

In [4]:
# 3. Create the Plotly Heatmap
fig = px.imshow(
    corr_matrix,
    text_auto=".2f", # Shows values with 2 decimal places
    aspect="auto",
    color_continuous_scale='RdBu_r', # Red-Blue scale (good for positive/negative)
    title="Perth Solar Grid: Feature Correlation Matrix",
    labels=dict(color="Correlation Score")
)

In [5]:
# 4. Save as an Artifact (Professional move!)
fig.write_html(r"D:\Saqib\SkillSetExpand\WA_Solar_Grid_Optimizer/artifacts/correlation_heatmap.html")
fig.show()

Here is the breakdown of what your data is telling you:

1. The Strongest Link: Temperature (0.54)
There is a moderate positive correlation (0.54) between temperature_2m and direct_normal_irradiance.

The Interpretation: Generally, as it gets hotter in Perth, the sun is shining more directly.

The "Muscle" Insight: In an interview, mention that while temperature is a good proxy for sunshine, it can be a double-edged sword. Solar panels actually lose efficiency as they get extremely hot (over 25°C). Your model will need to learn this non-linear relationship.

2. The Cloud "Choke" (-0.28)
There is a weak-to-moderate negative correlation (-0.28) between cloud_cover and direct_normal_irradiance.

The Interpretation: As cloud cover increases, the direct sun hitting the panels decreases.

The Surprise: You might expect this number to be much closer to -1.0. Why is it only -0.28?

Perth Weather: Perth often has "high-level" thin clouds or scattered clouds that don't completely block the sun.

Time of Day: At night, cloud cover can be 100%, but irradiance is 0 regardless. This "nighttime noise" dilutes the correlation score.

3. Temperature vs. Clouds (-0.15)
There is almost no significant relationship (-0.15) between clouds and temperature in this specific dataset.

The Interpretation: This is great for your machine learning model! It means cloud_cover and temperature provide independent information. In ML terms, you don't have a "multi-collinearity" problem here—both features are useful and aren't just repeating each other.

Your Next "Brain" Move: Time-Series Seasonality
The correlation matrix is a "snapshot" of the whole year, but Perth's weather is famously seasonal. A correlation of 0.54 might be 0.80 in the scorching January summer and 0.20 in a stormy July winter.

In [7]:
import plotly.graph_objects as go
df['time'] = pd.to_datetime(df['time'])

# 2. Resample to Monthly Average
# This smooths out the daily 'spikes' to show the seasonal 'wave'
monthly_df = df.resample('ME', on='time').mean().reset_index()



In [8]:
# 3. Create a Dual-Axis Plot (Professional Move!)
fig = go.Figure()

# Add Irradiance (Solar Power)
fig.add_trace(go.Scatter(
    x=monthly_df['time'],
    y=monthly_df['direct_normal_irradiance'],
    name="Avg Solar Irradiance",
    line=dict(color='orange', width=4)
))

# Add Temperature on a secondary Y-axis
fig.add_trace(go.Scatter(
    x=monthly_df['time'],
    y=monthly_df['temperature_2m'],
    name="Avg Temperature (°C)",
    line=dict(color='red', dash='dot'),
    yaxis="y2"
))

# 4. Layout for Double Y-Axis
fig.update_layout(
    title="Perth Solar Grid: Seasonal Energy vs. Temperature Trend",
    xaxis_title="Month",
    yaxis_title="Irradiance (W/m²)",
    yaxis2=dict(title="Temperature (°C)", overlaying="y", side="right"),
    template="plotly_dark"
)



In [9]:
# 5. Save the Artifact
fig.write_html(r"D:\Saqib\SkillSetExpand\WA_Solar_Grid_Optimizer\artifacts/seasonal_trend.html")
fig.show()

Here is the "Brain" analysis of your plot that you should mention in your project documentation:

1. The "Lag" Insight (March – May)
Look at the start of your plot. Temperature (red dots) actually peaks in March, but Irradiance (orange line) is already in a steep decline.

The "Muscle" Explanation: This proves that temperature alone is a "lagging" indicator. The sun’s angle (declination) is changing faster than the earth is cooling down. If you built a simple linear model, it would drastically over-estimate power in Autumn.

2. The July "Floor"
Both lines bottom out in July (the heart of WA winter).

The Prediction Insight: This is your most stable period for prediction. The variance is low because it’s consistently cooler and cloudier. Your model will likely have its lowest error rates here.

3. The October "Solar Jump"
Notice the sharp vertical spike in Irradiance around October/November while temperature stays relatively flat.

The "Muscle" Explanation: This represents the clear-sky "spring surge" in Perth. This is where your Cloud Cover feature (from your correlation matrix) will do the heavy lifting to explain that sudden jump in energy.